# copyme prototype
1. parse personal writing as context
2. process text to structured description of linguistic style

e.g.
```python
linguistic_style_metrics = {
    "Type-Token Ratio (TTR)": {
        "description": "The ratio of unique words to the total word count, used to measure vocabulary diversity and lexical richness.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0}
    },
    "Mean Length of Sentence (MLS)": {
        "description": "The average number of words per sentence, indicating the syntactic complexity and density of a text.",
        "data_type": "float",
        "constraints": {"min": 1.0}
    },
    "Punctuation Density": {
        "description": "The frequency of specific marks (e.g., semi-colons, dashes) per 1,000 words, reflecting pacing and structural sophistication.",
        "data_type": "float",
        "constraints": {"min": 0.0}
    },
    "Discourse Marker Density": {
        "description": "The frequency of conversational fillers (e.g., 'so', 'actually', 'like') per 1,000 words. High values indicate a spontaneous, 'thinking-out-loud' speaking style.",
        "data_type": "float",
        "constraints": {"min": 0.0}
    },
    "Hapax Legomena Ratio": {
        "description": "The ratio of words that appear exactly once to the total unique vocabulary. High values suggest precise, non-repetitive description; low values suggest reliance on 'anchor' keywords.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0}
    },
    "Idiosyncrasy Score": {
        "description": "The percentage of the text composed of repeated 2-gram or 3-gram phrases. Measures reliance on habitual 'crutch phrases' or verbal tics.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0}
    },
    "Modal Hedging Ratio": {
        "description": "The ratio of 'soft' modals (could, might, try) to total modal verbs. A value of 1.0 indicates pure uncertainty/suggestion; 0.0 indicates pure authority/certainty.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0}
    },
    "Function Word Frequency": {
        "description": "The distribution of particles like prepositions and conjunctions, which reveals subconscious patterns in an author's 'linguistic fingerprint'.",
        "data_type": "dict[str, float]",
        "constraints": {"value_min": 0.0, "value_max": 1.0}
    },
    "Readability Index": {
        "description": "A score derived from syllable counts and sentence lengths (e.g., Flesch-Kincaid) to determine the text's accessibility level.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 100.0}
    },
    "Lexical Sophistication": {
        "description": "A qualitative assessment of the use of low-frequency, technical, or academic terminology within a specific context.",
        "data_type": "str",
        "allowed_values": ["low", "medium", "high", "very high"]
    },
    "Syntactic Variety": {
        "description": "The diversity of sentence types and structures (e.g., active vs. passive voice) that contribute to the rhythm of the writing.",
        "data_type": "str",
        "allowed_values": ["minimal", "moderate", "diverse", "highly diverse"]
    },
    "Cohesive Harmony": {
        "description": "The effectiveness of logical connectors and transitions that link ideas across paragraphs to ensure a smooth flow.",
        "data_type": "str",
        "allowed_values": ["weak", "adequate", "strong", "excellent"]
    },
    "Rhetorical Intent": {
        "description": "A qualitative evaluation of the author's purpose, such as persuasion or exposition, and how it influences the tone of the piece.",
        "data_type": "str",
        "allowed_values": ["informative", "persuasive", "expository", "narrative", "descriptive", "argumentative"]
    },
    "Genre Alignment": {
        "description": "The degree to which the writing style adheres to established conventions of a specific field, such as academic, legal, or literary standards.",
        "data_type": "str",
        "allowed_values": ["academic", "legal", "literary", "journalistic", "technical", "conversational"]
    }
}



```

In [1]:
import spacy
import textstat
from empath import Empath
from collections import Counter

## Prototype example of generating linquistic metrics

In [2]:
import spacy
import textstat
from empath import Empath
from collections import Counter

# --- INITIALIZATION ---
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Error: Model not found. Run: python -m spacy download en_core_web_sm")
    exit()

lexicon = Empath()

def analyze_linguistic_style(text):
    """
    Analyzes text to return a dictionary of quantitative metrics 
    and qualitative heuristics (using Empath for tone/intent).
    """
    doc = nlp(text)
    
    # --- 1. BASIC PRE-PROCESSING ---
    words = [token.text for token in doc if not token.is_punct and not token.is_space]
    word_count = len(words)
    sentences = list(doc.sents)
    sentence_count = len(sentences)
    
    if word_count == 0: return None

    # --- 2. QUANTITATIVE METRICS ---
    
    # Type-Token Ratio (TTR)
    unique_words = set(word.lower() for word in words)
    ttr = len(unique_words) / word_count

    # Mean Length of Sentence (MLS)
    mls = word_count / sentence_count

    # Punctuation Density (per 1000 words)
    punct_count = len([token for token in doc if token.is_punct])
    punct_density = (punct_count / word_count) * 1000

    # Function Word Frequency
    pos_counts = Counter(token.pos_ for token in doc)
    total_pos = sum(pos_counts.values())
    function_word_freq = {
        "prepositions": round(pos_counts.get("ADP", 0) / total_pos, 3),
        "conjunctions": round((pos_counts.get("CCONJ", 0) + pos_counts.get("SCONJ", 0)) / total_pos, 3),
        "pronouns": round(pos_counts.get("PRON", 0) / total_pos, 3),
        "determiners": round(pos_counts.get("DET", 0) / total_pos, 3)
    }

    # Readability Index (Flesch Reading Ease: 0-100)
    readability = textstat.flesch_reading_ease(text)

    # --- 3. QUALITATIVE ATTRIBUTES (via EMPATH & Heuristics) ---

    # Analyze text topics with Empath
    # 'normalize=True' gives us a ratio (e.g., 0.10 means 10% of words match category)
    empath_scores = lexicon.analyze(text, normalize=True)

    # A. Rhetorical Intent
    # We map Empath categories to intentions
    if empath_scores['help'] > 0.02 or empath_scores['negotiate'] > 0.02:
        intent = "persuasive"
    elif empath_scores['science'] > 0.02 or empath_scores['school'] > 0.02:
        intent = "expository" # educational
    elif empath_scores['emotional'] > 0.02 or empath_scores['negative_emotion'] > 0.02:
        intent = "narrative" # storytelling
    else:
        intent = "informative" # default

    # B. Genre Alignment
    # Combining Readability + Empath Topics
    if readability < 40 and (empath_scores['law'] > 0.01 or empath_scores['government'] > 0.01):
        genre = "legal"
    elif readability < 50 and empath_scores['science'] > 0.02:
        genre = "academic"
    elif readability > 60 and intent == "narrative":
        genre = "literary"
    elif empath_scores['computer'] > 0.02 or empath_scores['technology'] > 0.02:
        genre = "technical"
    else:
        genre = "conversational"

    # C. Lexical Sophistication
    # Heuristic: Ratio of words with 7+ letters
    long_words = [w for w in words if len(w) > 6]
    long_word_ratio = len(long_words) / word_count
    
    if long_word_ratio > 0.3: sophistication = "very high"
    elif long_word_ratio > 0.2: sophistication = "high"
    elif long_word_ratio > 0.12: sophistication = "medium"
    else: sophistication = "low"

    # D. Syntactic Variety
    # Heuristic: Standard Deviation of sentence lengths
    sent_lengths = [len([t for t in s if not t.is_punct]) for s in sentences]
    # Simple variance calculation
    mean_len = sum(sent_lengths) / len(sent_lengths)
    variance = sum((x - mean_len) ** 2 for x in sent_lengths) / len(sent_lengths)
    std_dev = variance ** 0.5
    
    if std_dev > 10: variety = "highly diverse"
    elif std_dev > 6: variety = "diverse"
    elif std_dev > 3: variety = "moderate"
    else: variety = "minimal"

    # E. Cohesive Harmony
    # Heuristic: Noun overlap between sentences (Referential Cohesion)
    # Checks if adjacent sentences share at least one noun/proper noun
    overlaps = 0
    if sentence_count > 1:
        for i in range(1, sentence_count):
            prev_nouns = {t.lemma_ for t in sentences[i-1] if t.pos_ in ["NOUN", "PROPN"]}
            curr_nouns = {t.lemma_ for t in sentences[i] if t.pos_ in ["NOUN", "PROPN"]}
            if not prev_nouns.isdisjoint(curr_nouns):
                overlaps += 1
        cohesion_score = overlaps / (sentence_count - 1)
    else:
        cohesion_score = 1.0 # Single sentence is perfectly cohesive with itself

    if cohesion_score > 0.5: harmony = "strong"
    elif cohesion_score > 0.25: harmony = "adequate"
    else: harmony = "weak"


    # --- 4. COMPILE FINAL DICTIONARY ---
    return {
        "Type-Token Ratio (TTR)": round(ttr, 3),
        "Mean Length of Sentence (MLS)": round(mls, 2),
        "Punctuation Density": round(punct_density, 2),
        "Function Word Frequency": function_word_freq,
        "Readability Index": round(readability, 1),
        "Lexical Sophistication": sophistication,
        "Syntactic Variety": variety,
        "Cohesive Harmony": harmony,
        "Rhetorical Intent": intent,
        "Genre Alignment": genre
    }

# --- EXAMPLE USAGE ---

# Sample 1: Academic / Formal
text_academic = """
The proliferation of digital technologies has fundamentally altered the landscape of modern pedagogy. 
Specifically, the integration of algorithmic assessment tools allows for granular analysis of student performance. 
However, this reliance on quantitative metrics often obscures the qualitative nuances of learning. 
Thus, educators must maintain a critical distance from these tools.
"""

# Sample 2: Narrative / Simple
text_narrative = """
I walked down the street and saw a bright light. It was strange. 
I felt scared but I kept walking anyway because I wanted to see what it was. 
Suddenly, a cat jumped out! I laughed.
"""

print("--- ACADEMIC TEXT ANALYSIS ---")
import json
print(json.dumps(analyze_linguistic_style(text_academic), indent=4))

print("\n--- NARRATIVE TEXT ANALYSIS ---")
print(json.dumps(analyze_linguistic_style(text_narrative), indent=4))


--- ACADEMIC TEXT ANALYSIS ---
{
    "Type-Token Ratio (TTR)": 0.84,
    "Mean Length of Sentence (MLS)": 12.5,
    "Punctuation Density": 140.0,
    "Function Word Frequency": {
        "prepositions": 0.129,
        "conjunctions": 0.0,
        "pronouns": 0.0,
        "determiners": 0.113
    },
    "Readability Index": 4.6,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "conversational"
}

--- NARRATIVE TEXT ANALYSIS ---
{
    "Type-Token Ratio (TTR)": 0.806,
    "Mean Length of Sentence (MLS)": 7.2,
    "Punctuation Density": 166.67,
    "Function Word Frequency": {
        "prepositions": 0.043,
        "conjunctions": 0.065,
        "pronouns": 0.174,
        "determiners": 0.065
    },
    "Readability Index": 96.7,
    "Lexical Sophistication": "medium",
    "Syntactic Variety": "moderate",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "narra

In [ ]:
# Sample 3: Legal / Contractual
text_legal = """
The Party of the First Part shall, subject to the provisions herein contained, indemnify and hold harmless the Party of the Second Part against all claims, liabilities, and expenses arising from the breach of any representation or warranty made by the indemnifying party under this Agreement. This indemnification shall survive the termination of this Agreement for a period of five years.
"""

# Sample 4: Technical / Computer Science
text_technical = """
To optimize the neural network's performance, we implemented a custom gradient descent algorithm using TensorFlow and CUDA kernels. The architecture utilizes convolutional layers followed by a transformer block to capture long-range dependencies in the input data. We observed a significant reduction in latency after migrating the compute-intensive tasks to the GPU.
"""

# Sample 5: Journalistic / News
text_journalistic = """
City officials announced a new initiative today aimed at reducing traffic congestion in the downtown area. The plan includes the expansion of bike lanes and the introduction of a smart traffic management system. Residents expressed cautious optimism about the project, noting that similar efforts in the past had mixed results.
"""

print("--- LEGAL TEXT ANALYSIS ---")
import json
print(json.dumps(analyze_linguistic_style(text_legal), indent=4))

print("\n--- TECHNICAL TEXT ANALYSIS ---")
print(json.dumps(analyze_linguistic_style(text_technical), indent=4))

print("\n--- JOURNALISTIC TEXT ANALYSIS ---")
print(json.dumps(analyze_linguistic_style(text_journalistic), indent=4))

In [3]:
# --- EXTREME EXAMPLES ---

# 1. High Type-Token Ratio (TTR) - Every word is unique
text_extreme_ttr = "Linguistic diversity necessitates utilizing unique vocabulary, avoiding repetitive lexical selections, demonstrating expansive terminology."

# 2. High Mean Length of Sentence (MLS) - One extremely long, complex sentence
text_extreme_mls = "The multi-faceted nature of socioeconomic disparities within urban environments necessitates a comprehensive re-evaluation of current infrastructural paradigms, particularly concerning the equitable distribution of public resources and the mitigation of systemic barriers that historically marginalized communities encounter on a daily basis while attempting to navigate bureaucratic hurdles and access essential services that are often gatekept by opaque institutional frameworks."

# 3. High Punctuation Density - Overuse of various marks
text_extreme_punct = "Wait; what? Is this--really--necessary?! (Maybe...) Yes: absolutely, definitely, surely; however, why???"

# 4. Low Readability (Hard to read) - High syllables, complex structure
text_extreme_hard_read = "The phenomenological investigation of existential angst requires a profound ontological commitment to deciphering the primordial structures of human consciousness and its inherent teleological orientation toward transcendence within a secularized milieu."

# 5. High Syntactic Variety - Extremely different sentence lengths
text_extreme_syntax_variety = "Stop. Now, let us consider the profound implications of our actions on the future of this planet. Think. We must act before it is too late, for the consequences of our current trajectory are nothing short of catastrophic for every living being on Earth. Now!"

# 6. High Cohesive Harmony - Strong noun overlap
text_extreme_harmony = "The architect designed the building. This building features a unique glass facade. The facade reflects the surrounding city skyline. The skyline defines the character of the urban landscape."

# 7. High Persuasive Intent (Rhetorical)
text_extreme_persuasive = "We must stand together! We need your help to win this fight. Join us now, negotiate for a better future, and help us transform the world today!"

samples = {
    "EXTREME TTR": text_extreme_ttr,
    "EXTREME MLS": text_extreme_mls,
    "EXTREME PUNCTUATION": text_extreme_punct,
    "EXTREME HARD READ": text_extreme_hard_read,
    "EXTREME SYNTACTIC VARIETY": text_extreme_syntax_variety,
    "EXTREME HARMONY": text_extreme_harmony,
    "EXTREME PERSUASIVE": text_extreme_persuasive
}

for name, text in samples.items():
    print(f"--- {name} ---")
    print(json.dumps(analyze_linguistic_style(text), indent=4))
    print()

--- EXTREME TTR ---
{
    "Type-Token Ratio (TTR)": 1.0,
    "Mean Length of Sentence (MLS)": 13.0,
    "Punctuation Density": 230.77,
    "Function Word Frequency": {
        "prepositions": 0.0,
        "conjunctions": 0.0,
        "pronouns": 0.0,
        "determiners": 0.0
    },
    "Readability Index": -112.2,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "strong",
    "Rhetorical Intent": "informative",
    "Genre Alignment": "conversational"
}

--- EXTREME MLS ---
{
    "Type-Token Ratio (TTR)": 0.867,
    "Mean Length of Sentence (MLS)": 60.0,
    "Punctuation Density": 66.67,
    "Function Word Frequency": {
        "prepositions": 0.109,
        "conjunctions": 0.047,
        "pronouns": 0.031,
        "determiners": 0.078
    },
    "Readability Index": -72.3,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "strong",
    "Rhetorical Intent": "informative",
    "Ge

In [ ]:
nick_wong_cover_letter = """I recently talked to you at Chiasma Synapse on the 21/9/2016, where you and your colleague
were recruiting students who are passionate about gaining science industry experience.
I have spent this year in the lab training in molecular biology techniques related to cancer
research as part of my honours degree. I have also conducted RNA extraction and RT-qPCR
in two of my 300 level papers during my undergraduate degree. I am very resourceful, and I
constantly think of more efficient ways to complete tasks.
I initially became interested in science industry from a talk you gave in the biochemistry
department last year. Since then, I have gained insight regarding the status quo between
academia and the industry. It is evident that academia portrays a strong prestige over the
industry. This led me to approach someone that had experience in both fields, one of your
founders, Parry Guilford. Parry deeply inspired me, and as a result, I have taken up the
responsibility of being CEO for Chiasma Dunedin in 2017.
I jumped at the opportunity to apply for this internship because I could see the invaluable
experience I would gain. I believe this will be a strong kick start to my career, which will
allow me to pursue my dreams."""

print(json.dumps(analyze_linguistic_style(nick_wong_cover_letter), indent=4))



{
    "Type-Token Ratio (TTR)": 0.648,
    "Mean Length of Sentence (MLS)": 19.09,
    "Punctuation Density": 95.24,
    "Function Word Frequency": {
        "prepositions": 0.115,
        "conjunctions": 0.037,
        "pronouns": 0.131,
        "determiners": 0.066
    },
    "Readability Index": 49.1,
    "Lexical Sophistication": "high",
    "Syntactic Variety": "moderate",
    "Cohesive Harmony": "adequate",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "academic"
}


In [8]:
ai_reversed_engine_text = """I am writing to express my enthusiastic interest in the Biotechnology Research Internship position at Pacific Flair, as advertised on your professional careers portal. My academic background in molecular biology, coupled with a rigorous focus on genomic sequencing, aligns with the innovative research priorities established by your organisation. Throughout my undergraduate studies, I have maintained a steadfast commitment to understanding the biochemical pathways that facilitate cellular regeneration; consequently, I am eager to apply this theoretical knowledge within a practical, high-throughput laboratory environment.
During my recent laboratory assistantship, I prioritised the meticulous optimisation of PCR protocols to ensure the integrity of diverse genetic samples. This role required a high degree of technical precision and an ability to synthesise complex data sets into coherent reports for senior researchers. The experience refined my competency in utilising CRISPR-Cas9 technologies; furthermore, it underscored the importance of methodological consistency when navigating the challenges inherent in longitudinal clinical trials. I am confident that my familiarity with these advanced analytical frameworks will allow me to contribute meaningfully to your ongoing developmental projects.
Pacific Flair’s reputation for pioneering sustainable biopharmaceuticals represents the ideal environment for an aspiring scientist dedicated to translational medicine. I am particularly drawn to your recent publications regarding enzymatic catalysis, which demonstrate a sophisticated approach to solving contemporary therapeutic hurdles. I would welcome the opportunity to discuss how my technical skills and academic dedication can support your team's objectives during a formal interview. Thank you for your time and for considering my application.
"""

print(json.dumps(analyze_linguistic_style(ai_reversed_engine_text), indent=4))

{
    "Type-Token Ratio (TTR)": 0.709,
    "Mean Length of Sentence (MLS)": 22.82,
    "Punctuation Density": 95.62,
    "Function Word Frequency": {
        "prepositions": 0.108,
        "conjunctions": 0.025,
        "pronouns": 0.094,
        "determiners": 0.076
    },
    "Readability Index": -7.1,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "moderate",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "academic"
}


In [9]:
ai_reversed_engine_text_v2 = """I am writing to apply for the Biotechnology Research Internship at Pacific Flair, as advertised on your professional careers website. My academic background in molecular biology, coupled with genomic sequencing, aligns with the innovative research goals and priorities of your organisation. Throughout my undergraduate studies, I maintained a consistent focus on the biochemical pathways that facilitate cellular regeneration within various clinical settings. Consequently, I am eager to apply this knowledge within your high-throughput laboratory environment to support your current research objectives.

In my recent assistantship, I prioritised the optimisation of PCR protocols; this ensured the long-term integrity of our genetic samples. This role required high technical precision and the ability to synthesise complex data into reports for the senior staff. Additionally, it allowed me to refine my skills in CRISPR technologies while I followed strict guidelines for various laboratory trials. Furthermore, I am certain that my familiarity with these analytical frameworks will help me contribute to your latest developmental projects.

Pacific Flair is known for pioneering sustainable biopharmaceuticals, which represents an ideal environment for an aspiring and dedicated scientist. I am particularly interested in your work regarding enzymatic catalysis, because it addresses many contemporary therapeutic hurdles through science. I look forward to discussing how my technical skills and my personal dedication can benefit your professional research team. Thank you for your time, and I appreciate your consideration of my application to join your internship programme."""

print(json.dumps(analyze_linguistic_style(ai_reversed_engine_text_v2), indent=4))

{
    "Type-Token Ratio (TTR)": 0.667,
    "Mean Length of Sentence (MLS)": 19.75,
    "Punctuation Density": 109.7,
    "Function Word Frequency": {
        "prepositions": 0.102,
        "conjunctions": 0.038,
        "pronouns": 0.136,
        "determiners": 0.049
    },
    "Readability Index": 12.0,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "academic"
}


In [10]:
my_custom_cover_letter_overly_excited = """
Hi team at leonardo! I have been an avid user of your ai generation tool for years now, and it is a product that I absolutely love!
I have use it extensively in my life, and it has speed up a lot of my work, especially when it comes to front end design assets. 
I see that you have an opening for a front end engineer, which is a recent career path I have decided to take. I believe that my background in
graphic design and machine learning would provide the diverse background that will enable me to excel in this role long term. 
I absolutely love learning, and continue to learn new things every day. I look to you reviewing my application. Kind regards, 
Josh"""

print(json.dumps(analyze_linguistic_style(my_custom_cover_letter_overly_excited), indent=4))


{
    "Type-Token Ratio (TTR)": 0.659,
    "Mean Length of Sentence (MLS)": 15.75,
    "Punctuation Density": 103.17,
    "Function Word Frequency": {
        "prepositions": 0.076,
        "conjunctions": 0.048,
        "pronouns": 0.159,
        "determiners": 0.062
    },
    "Readability Index": 70.7,
    "Lexical Sophistication": "medium",
    "Syntactic Variety": "diverse",
    "Cohesive Harmony": "adequate",
    "Rhetorical Intent": "informative",
    "Genre Alignment": "conversational"
}


In [15]:
teenager_vibe = """
Okay, so we had this data platform that we really wanted to get research data on, and so it was a batch job, and I really didn't know what to actually do at first, so we just had to weigh it manually to try to figure out the actual pain points that we needed to address, and we found that we don't really need any massive parallel Spark processing information. What we really need is a way to do release management and a way to manage schema deployments, schema changes, and do validation, do it all in a way that we could like trace it.
Oh my god, so I don't actually know too much about privacy and compliance, but we did try to make sure everything was secured in a VPC and that we had sufficient security checks and fine-grained access, so you couldn't have any Joe blog just come in and try to access the data and have to go through us.
"""

print(json.dumps(analyze_linguistic_style(teenager_vibe), indent=4))


{
    "Type-Token Ratio (TTR)": 0.538,
    "Mean Length of Sentence (MLS)": 56.33,
    "Punctuation Density": 88.76,
    "Function Word Frequency": {
        "prepositions": 0.043,
        "conjunctions": 0.075,
        "pronouns": 0.134,
        "determiners": 0.053
    },
    "Readability Index": 34.2,
    "Lexical Sophistication": "medium",
    "Syntactic Variety": "highly diverse",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "informative",
    "Genre Alignment": "technical"
}


In [12]:
import nltk

# Trying version 2, with more personal stuff

In [24]:
import spacy
import nltk
import textstat
from empath import Empath
from collections import Counter

# --- INITIALIZATION ---

# 1. Load Spacy Model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Error: Spacy model not found. Run: python -m spacy download en_core_web_sm")
    exit()

# 2. Load NLTK Resources
try:
    from nltk.corpus import nps_chat
    try:
        chat_words = nps_chat.tagged_words()
        NLTK_FILLERS = {word.lower() for word, tag in chat_words if tag == 'UH'}
    except LookupError:
        print("Warning: NLTK 'nps_chat' not found. Using fallback list.")
        NLTK_FILLERS = set()
except ImportError:
    NLTK_FILLERS = set()

FALLBACK_MARKERS = {
    "so", "actually", "really", "okay", "well", "like", "basically", 
    "literally", "totally", "honestly", "anyway", "you know", "i mean", "right"
}
DISCOURSE_MARKERS = NLTK_FILLERS.union(FALLBACK_MARKERS)

lexicon = Empath()


# --- MODULE 1: PREPROCESSING ---

def preprocess_text(text):
    doc = nlp(text)
    words = [token.text for token in doc if not token.is_punct and not token.is_space]
    sentences = list(doc.sents)
    
    return {
        "doc": doc,
        "words": words,
        "word_count": len(words),
        "sentences": sentences,
        "sentence_count": len(sentences),
        "unique_words": set(w.lower() for w in words)
    }


# --- MODULE 2: STANDARD QUANTITATIVE METRICS ---

def get_ttr(data):
    if data["word_count"] == 0: return 0.0
    return round(len(data["unique_words"]) / data["word_count"], 3)

def get_mls(data):
    if data["sentence_count"] == 0: return 0.0
    return round(data["word_count"] / data["sentence_count"], 2)

def get_punctuation_density(data):
    if data["word_count"] == 0: return 0.0
    punct_count = len([t for t in data["doc"] if t.is_punct])
    return round((punct_count / data["word_count"]) * 1000, 2)

def get_readability(text):
    return round(textstat.flesch_reading_ease(text), 1)

def calculate_standard_metrics(data, text):
    return {
        "Type-Token Ratio": get_ttr(data),
        "Mean Length Sentence": get_mls(data),
        "Punctuation Density": get_punctuation_density(data),
        "Readability Index": get_readability(text)
    }


# --- MODULE 3: PERSONAL VOICE MARKERS ---

def get_discourse_marker_density(data):
    if data["word_count"] == 0: return 0.0
    dm_count = sum(1 for w in data["words"] if w.lower() in DISCOURSE_MARKERS)
    return round((dm_count / data["word_count"]) * 1000, 2)

def get_hapax_legomena_ratio(data):
    total_unique = len(data["unique_words"])
    if total_unique == 0: return 0.0
    word_freqs = Counter(w.lower() for w in data["words"])
    hapax_count = sum(1 for count in word_freqs.values() if count == 1)
    return round(hapax_count / total_unique, 3)

def get_common_vocabulary(data, n=10):
    word_freqs = Counter(w.lower() for w in data["words"])
    return [word for word, count in word_freqs.most_common(n)]

# -- Idiosyncrasy Helpers --
def get_ngrams(words, n):
    return [" ".join(words[i:i+n]).lower() for i in range(len(words)-n+1)]

def get_repeated_ngram_counts(words, n_range=[2, 3]):
    combined_counts = Counter()
    for n in n_range:
        ngrams = get_ngrams(words, n)
        counts = Counter(ngrams)
        # Only keep repeated ones
        for gram, count in counts.items():
            if count > 1:
                combined_counts[gram] += count
    return combined_counts

def get_idiosyncrasy_score(word_count, repeated_counts):
    if word_count == 0: return 0.0
    total_repeated = sum(repeated_counts.values())
    return round(total_repeated / word_count, 4)

def get_frequent_phrases(repeated_counts):
    return list(repeated_counts.keys())

# -- Modal Helpers --
def get_modal_ratio(data):
    modals = [token.lemma_.lower() for token in data["doc"] if token.tag_ == "MD"]
    if not modals: return 0.5
    
    strong_modals = {"must", "will", "shall", "have", "need", "ought"}
    weak_modals = {"can", "could", "may", "might", "would", "should"}
    
    weak_count = sum(1 for m in modals if m in weak_modals)
    total_modals = len(modals)
    
    return round(weak_count / total_modals, 2)

def calculate_voice_markers(data):
    # Calculate intermediate data for idiosyncrasy
    repeated_counts = get_repeated_ngram_counts(data["words"])
    
    return {
        "Discourse Marker Density": get_discourse_marker_density(data),
        "Hapax Legomena Ratio": get_hapax_legomena_ratio(data),
        "Idiosyncrasy Score": get_idiosyncrasy_score(data["word_count"], repeated_counts),
        "Modal Hedging Ratio": get_modal_ratio(data),
        "Frequent Phrases": get_frequent_phrases(repeated_counts),
        "Common Vocabulary": get_common_vocabulary(data)
    }


# --- MODULE 4: POS & FUNCTION WORDS ---

def get_pos_counts(doc):
    return Counter(token.pos_ for token in doc)

def get_pos_ratio(pos_counts, target_pos_list):
    total = sum(pos_counts.values()) or 1
    count = sum(pos_counts.get(pos, 0) for pos in target_pos_list)
    return round(count / total, 3)

def calculate_pos_frequencies(data):
    doc = data["doc"]
    counts = get_pos_counts(doc)
    return {
        "prepositions": get_pos_ratio(counts, ["ADP"]),
        "conjunctions": get_pos_ratio(counts, ["CCONJ", "SCONJ"]),
        "pronouns": get_pos_ratio(counts, ["PRON"]),
        "determiners": get_pos_ratio(counts, ["DET"])
    }


# --- MODULE 5: QUALITATIVE PROFILE ---

def get_lexical_sophistication(data):
    words = data["words"]
    if not words: return "low"
    long_words = [w for w in words if len(w) > 6]
    ratio = len(long_words) / len(words)
    
    if ratio > 0.3: return "very high"
    elif ratio > 0.2: return "high"
    elif ratio > 0.12: return "medium"
    return "low"

def get_syntactic_variety(data):
    sentences = data["sentences"]
    if not sentences: return "minimal"
    
    sent_lengths = [len([t for t in s if not t.is_punct]) for s in sentences]
    if not sent_lengths: return "minimal"
    
    mean_len = sum(sent_lengths) / len(sent_lengths)
    variance = sum((x - mean_len) ** 2 for x in sent_lengths) / len(sent_lengths)
    std_dev = variance ** 0.5
    
    if std_dev > 10: return "highly diverse"
    elif std_dev > 6: return "diverse"
    elif std_dev > 3: return "moderate"
    return "minimal"

def get_cohesive_harmony(data):
    sentences = data["sentences"]
    if len(sentences) <= 1: return "strong"
    
    overlaps = 0
    for i in range(1, len(sentences)):
        prev_nouns = {t.lemma_ for t in sentences[i-1] if t.pos_ in ["NOUN", "PROPN"]}
        curr_nouns = {t.lemma_ for t in sentences[i] if t.pos_ in ["NOUN", "PROPN"]}
        if not prev_nouns.isdisjoint(curr_nouns):
            overlaps += 1
            
    score = overlaps / (len(sentences) - 1)
    
    if score > 0.5: return "strong"
    elif score > 0.25: return "adequate"
    return "weak"

def get_rhetorical_intent(text):
    empath_scores = lexicon.analyze(text, normalize=True) or {}
    if empath_scores.get('help', 0) > 0.02 or empath_scores.get('negotiate', 0) > 0.02:
        return "persuasive"
    elif empath_scores.get('science', 0) > 0.02 or empath_scores.get('school', 0) > 0.02:
        return "expository"
    elif empath_scores.get('emotional', 0) > 0.02 or empath_scores.get('negative_emotion', 0) > 0.02:
        return "narrative"
    return "informative"

def get_genre_alignment(text, intent):
    readability = textstat.flesch_reading_ease(text)
    empath_scores = lexicon.analyze(text, normalize=True) or {}
    
    if readability < 40 and (empath_scores.get('law', 0) > 0.01 or empath_scores.get('government', 0) > 0.01):
        return "legal"
    elif readability < 50 and empath_scores.get('science', 0) > 0.02:
        return "academic"
    elif readability > 60 and intent == "narrative":
        return "literary"
    elif empath_scores.get('computer', 0) > 0.02 or empath_scores.get('technology', 0) > 0.02:
        return "technical"
    return "conversational"

def calculate_qualitative_profile(data, text):
    intent = get_rhetorical_intent(text)
    return {
        "Lexical Sophistication": get_lexical_sophistication(data),
        "Syntactic Variety": get_syntactic_variety(data),
        "Cohesive Harmony": get_cohesive_harmony(data),
        "Rhetorical Intent": intent,
        "Genre Alignment": get_genre_alignment(text, intent)
    }


# --- FINAL CONSTRUCTOR ---

def analyze_text_style(text):
    data = preprocess_text(text)
    if data["word_count"] == 0:
        return {"error": "Text is empty."}

    return {
        "Standard Metrics": calculate_standard_metrics(data, text),
        "Personal Voice Markers": calculate_voice_markers(data),
        "POS Frequency": calculate_pos_frequencies(data),
        "Qualitative Profile": calculate_qualitative_profile(data, text)
    }


# --- USAGE EXAMPLE ---
if __name__ == "__main__":
    sample_text = """
    Okay, so we had this data platform that we really wanted to get research data on, and so it was a batch job, 
    and I really didn't know what to actually do at first, so we just had to weigh it manually to try to figure out 
    the actual pain points that we needed to address.
    """
    
    import json
    print(json.dumps(analyze_text_style(sample_text), indent=4))


{
    "Standard Metrics": {
        "Type-Token Ratio": 0.719,
        "Mean Length Sentence": 28.5,
        "Punctuation Density": 87.72,
        "Readability Index": 35.2
    },
    "Personal Voice Markers": {
        "Discourse Marker Density": 192.98,
        "Hapax Legomena Ratio": 0.78,
        "Idiosyncrasy Score": 0.0702,
        "Modal Hedging Ratio": 0.5,
        "Frequent Phrases": [
            "so we",
            "that we"
        ],
        "Common Vocabulary": [
            "to",
            "we",
            "so",
            "had",
            "data",
            "that",
            "really",
            "and",
            "it",
            "okay"
        ]
    },
    "POS Frequency": {
        "prepositions": 0.045,
        "conjunctions": 0.03,
        "pronouns": 0.152,
        "determiners": 0.045
    },
    "Qualitative Profile": {
        "Lexical Sophistication": "low",
        "Syntactic Variety": "highly diverse",
        "Cohesive Harmony": "weak",
        "

In [25]:
teenager_vibe = """
Okay, so we had this data platform that we really wanted to get research data on, and so it was a batch job, and I really didn't know what to actually do at first, so we just had to weigh it manually to try to figure out the actual pain points that we needed to address, and we found that we don't really need any massive parallel Spark processing information. What we really need is a way to do release management and a way to manage schema deployments, schema changes, and do validation, do it all in a way that we could like trace it.
Oh my god, so I don't actually know too much about privacy and compliance, but we did try to make sure everything was secured in a VPC and that we had sufficient security checks and fine-grained access, so you couldn't have any Joe blog just come in and try to access the data and have to go through us.
"""

print(json.dumps(analyze_text_style(teenager_vibe), indent=4))


{
    "Standard Metrics": {
        "Type-Token Ratio": 0.538,
        "Mean Length Sentence": 56.33,
        "Punctuation Density": 88.76,
        "Readability Index": 34.2
    },
    "Personal Voice Markers": {
        "Discourse Marker Density": 153.85,
        "Hapax Legomena Ratio": 0.681,
        "Idiosyncrasy Score": 0.1598,
        "Modal Hedging Ratio": 1.0,
        "Frequent Phrases": [
            "so we",
            "we had",
            "that we",
            "we really",
            "try to",
            "do n't",
            "really need",
            "a way",
            "way to",
            "in a",
            "a way to"
        ],
        "Common Vocabulary": [
            "to",
            "we",
            "and",
            "do",
            "so",
            "that",
            "a",
            "really",
            "it",
            "n't"
        ]
    },
    "POS Frequency": {
        "prepositions": 0.043,
        "conjunctions": 0.075,
        "pronouns": 0.

In [26]:
unconfident_fisherman = """
Dear Mr Captain, or whoever it is yourself that’s reading this,

Sorry to be bothering you with a letter, but I’m only after seeing your notice for a deckhand, and I thought I might chance my arm sending this in. To tell you the truth, I’ve been staring at the paper for an hour wondering if I should write at all, as I know a big boat like yours is serious business, and I wouldn’t want you thinking I’m trying to be something I’m not. I’m from just outside Dingle, a small spot really, and while I’ve spent a fair bit of time on my uncle’s punt, that’s only a small wooden thing for potting near the shore, not a proper vessel like the one you have, which looks fierce impressive, so it does.

I’m young enough, I’m only after finishing my school days, and this would be my first job, my first real one anyway. I’m a quiet lad, I don’t have much to say for myself usually, but I’m grand with the wet and the cold, sure being from Kerry I’m fairly used to being soaked to the bone. I might be a bit shaky on my sea legs starting off, and I’m sorry in advance if I ask a foolish question or two, but I swear on my life I’ll listen to whatever you tell me. I’m not one for messing about, and I’ve a good pair of hands on me, or so my Mam tells me anyway.

I’d be happy enough just scrubbing the deck or making the tea until I learn the ropes, I wouldn’t be looking to get in anyone’s way. I just want to work, and I’d be terrible grateful if you’d give a young fella a go. Thanks a million for your time, sorry again for the long note.

Yours,

Seán O’Connor"""

print(json.dumps(analyze_text_style(unconfident_fisherman), indent=4))


{
    "Standard Metrics": {
        "Type-Token Ratio": 0.519,
        "Mean Length Sentence": 30.45,
        "Punctuation Density": 110.45,
        "Readability Index": 70.4
    },
    "Personal Voice Markers": {
        "Discourse Marker Density": 116.42,
        "Hapax Legomena Ratio": 0.707,
        "Idiosyncrasy Score": 0.194,
        "Modal Hedging Ratio": 0.6,
        "Frequent Phrases": [
            "that \u2019s",
            "to be",
            "but i",
            "i \u2019m",
            "\u2019m only",
            "only after",
            "and i",
            "i might",
            "i \u2019ve",
            "if i",
            "i would",
            "would n\u2019t",
            "\u2019m not",
            "a small",
            "on my",
            "my first",
            "anyway i",
            "i \u2019d",
            "\u2019d be",
            "but i \u2019m",
            "i \u2019m only",
            "\u2019m only after",
            "i would n\u2019t",
            "